In [ ]:
pip install opendatasets

In [ ]:
import opendatasets as od
train_tweets = od.download("https://www.kaggle.com/vkrahul/twitter-hate-speech?select=train_E6oV3lV.csv")

In [ ]:
#import the required libraries
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup

def review_to_words(review):
    '''Function that takes in the word and remove html parser and other symbols.
    Args: review, the sentence in the string format
    returns: cleansed text'''
    text = BeautifulSoup(review, "html.parser").get_text() # Remove HTML tags
    text = re.sub(r"[^a-zA-Z0-9]", " ", text.lower()) # Convert to lower case
    return text

def read_csv(file_path):
    '''Function that takes in the file and returns in the format cleaned and ready for feature learning
    Args: csv file
    returns: datarfame'''
    df = pd.read_csv(file_path)          #read in the file
    df = df.drop(['id'], axis=1)         #drop the unecessary columns
    df['tweet'] = df['tweet'].apply(lambda x:review_to_words(x))   #apply the above funtion to preporcess the word
    df = df[['tweet','label']]            #rearrange the columns
    return df       #returns dataframe

#Now, specify the path that file located
path = r'twitter-hate-speech/train_E6oV3lV.csv'

#apply the function and view the dataframe
df = read_csv(path)
df.head(10)


In [ ]:
df.describe()

In [ ]:
#Code for removing mistake words
replace_words = {'luv':'love','wud':'would','lyk':'like','wateva':'whatever','ttyl':'talk to you later',
               'kul':'cool','fyn':'fine','omg':'oh my god!','fam':'family','bruh':'brother',
               'cud':'could','fud':'food'} ## Need a huge dictionary

#we will pass a sentence to test our function
sentence = "I luv myself"
words = sentence.split()    #split the sentence into words

#replace the words in the sentence with the word in the dictionary if the the actual word itself.
reformed = [replace_words[word] if word in replace_words else word for word in words]
reformed = " ".join(reformed)    #join the splitted after reforming
print(reformed)     #print the reformed

In [ ]:
#apply the function on df['tweet']
df['tweet'] = df['tweet'].apply(lambda x : ' '.join(replace_words[word] if word in replace_words else word for word in x.split()))

#disply the reformed
df.head()

In [ ]:
pip install wordcloud

In [ ]:
#Import the libraries
from wordcloud import WordCloud
import matplotlib.pyplot as plt

#take all words in df['tweet'] with the labels 0 for visualization
normal_words = ' '.join([word for word in df['tweet'][df['label'] == 0]])
#set the wordcloud parameters
wordcloud = WordCloud(width = 800, height = 500, max_font_size = 110,max_words = 100).generate(normal_words)
print('Normal words')
#plot the graph
plt.figure(figsize= (12,8))
plt.imshow(wordcloud, interpolation = 'bilinear',cmap='viridis')
plt.axis('off')

In [ ]:
#take all words in df['tweet'] with the labels 1 for visualization
normal_words = ' '.join([word for word in df['tweet'][df['label'] == 1]])
#set the wordcloud parameters
wordcloud = WordCloud(width = 800, height = 500, max_font_size = 110,max_words = 100).generate(normal_words)
print('Normal words')
#plot the figure
plt.figure(figsize= (12,8))
plt.imshow(wordcloud, interpolation = 'bilinear',cmap='viridis')
plt.axis('off')

In [ ]:
#replace the word user with empty space
df['tweet'] = df['tweet'].apply(lambda x: x.replace('user',''))

#print the dataframe
df.head()

In [ ]:
df.label.value_counts().plot(kind='barh')   #bar plot with value counts for each labels
print(df.label.value_counts()) #print it the numbers as well

In [ ]:
#print the shape and null counts of the dataframe

print("Shape: ", df.shape)
print("Label null values count: ", df['label'].isna().sum())
print("Tweets null values count: ", df['tweet'].isna().sum())

In [ ]:
#divide the dataframe with each class labels
df_positive = df[df['label']==0]
df_negative = df[df['label']==1]

#sample only 2500 out of all from the label 0
df_positive = df_positive.sample(2500)

# append the df with label 1 with label 0  (replaced .append -> pd.concat)
df_positive = pd.concat([df_positive, df_negative], ignore_index=True)

#Shuffle the df
from sklearn.utils import shuffle
df = shuffle(df_positive)

#reset the index
df = df.reset_index(drop=True)
df.head()

In [ ]:
#print the label counts and see how many are there

print("The label counts: ", df.label.value_counts())

In [ ]:
# we will write this file into the csv file and can read it later to save memory
df.to_csv('train/df.csv')
del df

In [ ]:
pip install --upgrade pip

In [ ]:
pip install torchtext --no-cache-dir

In [ ]:
pip install spacy --no-cache-dir

In [ ]:
#Downloading the spacy tokenizer english library before executing the build_vocab function, helps in building the tokenizer.
#Otherwise, the function build_vocab will throw an error.
import spacy    
from spacy.cli.download import download
download('en_core_web_sm')

In [ ]:
#Load the library
spacy.load('en_core_web_sm')

In [ ]:

from collections import Counter
from types import SimpleNamespace
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import spacy, torch, pandas as pd

# load spaCy tokenizer (ensure you ran the spacy model download earlier)
nlp = spacy.load('en_core_web_sm')

class SimpleTweetsDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = list(labels)
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], int(self.labels[idx])

def build_simple_vocab(texts, min_freq=3, max_vocab=20000):
    ctr = Counter()
    for t in texts:
        for tok in nlp(t):
            ctr[tok.text] += 1
    common = [w for w,f in ctr.most_common(max_vocab) if f >= min_freq]
    itos = ['<pad>', '<unk>'] + common
    stoi = {w:i for i,w in enumerate(itos)}
    return stoi, itos

def collate_fn_factory(stoi, pad_idx=0, unk_idx=1, max_len=512):
    def collate(batch):
        texts, labels = zip(*batch)
        tokenized = [[tok.text for tok in nlp(t)] for t in texts]
        indexed = [[stoi.get(w, unk_idx) for w in toks][:max_len] for toks in tokenized]
        lengths = [len(x) for x in indexed]
        maxl = max(lengths) if lengths else 0
        padded = torch.full((len(indexed), maxl), pad_idx, dtype=torch.long)
        for i, seq in enumerate(indexed):
            padded[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)
        labels_t = torch.tensor(labels, dtype=torch.long)
        batch_obj = SimpleNamespace()
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        batch_obj.tweet = (padded.to(device), torch.tensor(lengths, dtype=torch.long).to(device))
        batch_obj.label = labels_t.to(device)
        return batch_obj
    return collate

def build_vocab(file_path, batch_size=64, min_freq=3, max_vocab=20000, max_len=512, test_size=0.3, random_state=2019):
    # load preprocessed df (expects columns 'tweet' and 'label')
    df_local = pd.read_csv(file_path)
    if 'Unnamed: 0' in df_local.columns:
        df_local = df_local.drop(columns=['Unnamed: 0'])
    texts = df_local['tweet'].astype(str)
    labels = df_local['label'].astype(int)
    X_train, X_val, y_train, y_val = train_test_split(texts, labels, test_size=test_size, stratify=labels, random_state=random_state)
    stoi, itos = build_simple_vocab(X_train, min_freq=min_freq, max_vocab=max_vocab)
    pad_idx = stoi.get('<pad>', 0)
    unk_idx = stoi.get('<unk>', 1)
    collate = collate_fn_factory(stoi, pad_idx=pad_idx, unk_idx=unk_idx, max_len=max_len)
    train_ds = SimpleTweetsDataset(X_train, y_train)
    val_ds = SimpleTweetsDataset(X_val, y_val)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate)
    len_text_vocab = len(stoi)
    word_dict = stoi
    GLOBAL_WORD_DICT = word_dict     # save your vocab globally
    return train_loader, val_loader, len_text_vocab, word_dict




We will provide the above function with path and in turn get the iterators and word dict.

In [ ]:
#defining the path of the file
path = 'train/df.csv'

#assigining it to a function to get iterators
train_it, test_it, len_text_vocab, word_dict = build_vocab(path)

In [ ]:
#get the number of batches in each of the iterators

print(f'- nr. of training examples {len(train_it)}')
print(f'- nr. of testing examples {len(test_it)}')

In [ ]:
#Import libraries
import torch
from torch import nn
import torch.nn.functional as F

import random, math, sys

#set the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#Basic self attention model
class SelfAttention(nn.Module):
    """
    Basic Self Attention module, Heart of transormer
    Args: nn.Module from Pytorch
    """

    def __init__(self, emb, heads=8, mask=False):
        """
        emb: Embedding dimension
        heads: 8, as in emb are divided on heads and trained parallel
        mask: masking is done on seq-to-seq models, but it is kept here for future model development
        """

        super().__init__()        #inherit the pytorch

        assert emb % heads == 0, f'Embedding dimension ({emb}) should be divisible by nr. of heads ({heads})'

        self.emb = emb       #embedding dimensions
        self.heads = heads   #number of heds to split the job
        self.mask = mask     #masking if needed

        s = emb // heads
        # - We will break the embedding into `heads` chunks and feed each to a different attention head

        self.tokeys    = nn.Linear(emb, emb, bias=False)
        self.toqueries = nn.Linear(emb, emb, bias=False)
        self.tovalues  = nn.Linear(emb, emb, bias=False)

        self.unifyheads = nn.Linear(emb, emb)

    def forward(self, x):

        b, t, e = x.size()
        h = self.heads
        assert e == self.emb, f'Input embedding dim ({e}) should match layer embedding dim ({self.emb})'

        s = e // h

        keys    = self.tokeys(x)
        queries = self.toqueries(x)
        values  = self.tovalues(x)

        keys    = keys.view(b, t, h, s)
        queries = queries.view(b, t, h, s)
        values  = values.view(b, t, h, s)

        # -- We first compute the k/q/v's on the whole embedding vectors, and then split into the different heads.
        #    See the following video for an explanation: https://youtu.be/KmAISyVvE1Y

        # Compute scaled dot-product self-attention

        # - fold heads into the batch dimension
        keys = keys.transpose(1, 2).contiguous().view(b * h, t, s)
        queries = queries.transpose(1, 2).contiguous().view(b * h, t, s)
        values = values.transpose(1, 2).contiguous().view(b * h, t, s)

        queries = queries / (e ** (1/4))
        keys    = keys / (e ** (1/4))
        # - Instead of dividing the dot products by sqrt(e), we scale the keys and values.
        #   This should be more memory efficient

        # - get dot product of queries and keys, and scale
        dot = torch.bmm(queries, keys.transpose(1, 2))

        assert dot.size() == (b*h, t, t)

        if self.mask: # mask out the upper half of the dot matrix, excluding the diagonal
            mask_(dot, maskval=float('-inf'), mask_diagonal=False)

        dot = F.softmax(dot, dim=2)
        # - dot now has row-wise self-attention probabilities

        # apply the self attention to the values
        out = torch.bmm(dot, values).view(b, h, t, s)

        # swap h, t back, unify heads
        out = out.transpose(1, 2).contiguous().view(b, t, s * h)

        return self.unifyheads(out)

class TransformerBlock(nn.Module):
    '''Transfomer Block are modules that carry self-attention modules, based on the number of
    blocks needed on the main module, it is build that number of times and the data are fed into the
    each of the blocks and concatenated at the end'''

    def __init__(self, emb, heads, mask, seq_length, ff_hidden_mult=4, dropout=0.0, attention_type='default', pos_embedding=None):
        super().__init__()

        if attention_type == 'default':
            self.attention = SelfAttention(emb, heads=heads, mask=mask)
        self.mask = mask

        self.norm1 = nn.LayerNorm(emb)
        self.norm2 = nn.LayerNorm(emb)

        self.ff = nn.Sequential(

            nn.Linear(emb, ff_hidden_mult * emb),
            nn.ReLU(),
            nn.Linear(ff_hidden_mult * emb, emb)
        )

        self.do = nn.Dropout(dropout)

    def forward(self, x):

        attended = self.attention(x)

        x = self.norm1(attended + x)

        x = self.do(x)

        fedforward = self.ff(x)

        x = self.norm2(fedforward + x)

        x = self.do(x)

        return x

class CTransformer(nn.Module):
    """
    Transformer for classifying sequences
    """

    def __init__(self, emb, heads, depth, seq_length, num_tokens, num_classes, max_pool=True, dropout=0.0, wide=False):
        """
        emb: Embedding dimension
        heads: nr. of attention heads
        depth: Number of transformer blocks
        seq_length: Expected maximum sequence length
        num_tokens: Number of tokens (usually words) in the vocabulary
        num_classes: Number of classes.
        max_pool: If true, use global max pooling in the last layer. If false, use global
                         average pooling.
        """
        super().__init__()

        self.num_tokens, self.max_pool = num_tokens, max_pool

        self.token_embedding = nn.Embedding(embedding_dim=emb, num_embeddings=num_tokens)
        self.pos_embedding = nn.Embedding(embedding_dim=emb, num_embeddings=seq_length)

        tblocks = []
        for i in range(depth):
            tblocks.append(
                TransformerBlock(emb=emb, heads=heads, seq_length=seq_length, mask=False, dropout=dropout))

        self.tblocks = nn.Sequential(*tblocks)

        self.toprobs = nn.Linear(emb, num_classes)

        self.do = nn.Dropout(dropout)



    def forward(self, x):
        """
        x: A batch by sequence length integer tensor of token indices.
        return: predicted log-probability vectors for each token based on the preceding tokens.
        """
        tokens = self.token_embedding(x)
        b, t, e = tokens.size()

        positions = self.pos_embedding(torch.arange(t, device=device))[None, :, :].expand(b, t, e)

        x = tokens + positions
        x = self.do(x)
        x = self.tblocks(x)
        x = x.max(dim=1)[0] if self.max_pool else x.mean(dim=1) # pool over the time dimension
        x = self.toprobs(x)

        return F.log_softmax(x, dim=1)

In [ ]:
# defining a function for train loop
def train(train_loader, test_loader, num_epoch, opt):
    '''The function that takes in the iterator, number of epochs and optimizer and return the
    classification loss andaccuracy metrics.
    Args: Train and test iterator, number of epochs, optimzer
    returns: predicted labels, actual labels'''
    seen = 0
    #initialize every epoch
    epoch_loss = 0
    total, correction= 0.0, 0.0

    for e in range(num_epoch):     #in the range of epochs specified
        print(f'\n epoch {e}')     #print the nth of epoch
        #train the model
        model.train(True)
        #load the batch
        for batch in tqdm.tqdm(train_loader):
            opt.zero_grad()
            #specify the input and label
            input = batch.tweet[0]
            label = batch.label
            #send the input tensors to the model
            out = model(input)
            #get the result
            output = out.argmax(dim=1)
            #calculate the loss function
            loss = F.nll_loss(out, label)
            loss.backward()
            opt.step()
            seen += input.size(0)
            #loss and accuracy
            total += float(input.size(0))
            correction += float((label == output).sum().item())
            epoch_loss += loss.item()
            print('classification/train-loss', float(loss.item()), seen)
        accuracy = correction / total
        print(f'-- {"training validation"} accuracy {accuracy*100}')

        with torch.no_grad():

            model.train(False)  #model train not needed
            tot, cor= 0.0, 0.0  #metrics for calculating the accuracy
            collect_pred=[]     #list the collectiong the prediction labels
            collect_label=[]    #list for the actual labels

            for batch in tqdm.tqdm(test_it):

                input = batch.tweet[0]
                label = batch.label
                out = model(input).argmax(dim=1)

                collect_pred.append(out.cpu().detach().numpy())   #append the prediction to the list
                collect_label.append(label.cpu().detach().numpy())  #append the actual label to the list
                tot += float(input.size(0))                         #total from the input
                cor += float((label == out).sum().item())           #the correct ones

            acc = cor / tot    #accuracy
            print(f'-- {"test validation"} accuracy {acc*100}')
    torch.save(model.state_dict(), 'saved_weights2.pt')    #save the model
    print("The model is saved")
    pred, label = list(collect_pred), list(collect_label)  #put the np arrays of prediction and label into the list
    pred, label = np.concatenate(pred, axis=0), np.concatenate(label, axis=0)  #concatenate all the arrays
    return pred, label    #return the prediction and labels for the evaluation
    pass


In [ ]:
#setup the hyperparameters:
emb=128
heads=8
depth = 6
seq_length = 512
num_tokens = len_text_vocab
NUM_CLS = 2

#BUild the model
model = CTransformer(emb=emb,
                    heads=heads,
                    depth=depth,
                    seq_length=seq_length,
                    num_tokens=num_tokens,
                    num_classes=NUM_CLS,
                    max_pool=True)
if torch.cuda.is_available():
    model.cuda()
#push to cuda if available
opt = torch.optim.Adam(lr=0.0001, params=model.parameters())

In [ ]:
#call the train loop
pred, label = train(train_it,test_it, 20, opt)

In [ ]:
from sklearn import metrics
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt


print(classification_report(label, pred, labels=[0,1]))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


cm = confusion_matrix(label, pred, labels=[0,1])

ax= plt.subplot()
sns.heatmap(cm, annot=True, fmt='g', ax=ax);  #annot=True to annotate cells, ftm='g' to disable scientific notation
# labels, title and ticks
ax.set_xlabel('Predicted labels');ax.set_ylabel('True labels');
ax.set_title('Confusion Matrix');
ax.xaxis.set_ticklabels(['0', '1']); ax.yaxis.set_ticklabels(['0', '1']);

In [ ]:
#load weights
path='saved_weights2.pt'
model.load_state_dict(torch.load(path));
model.eval();

#Instantiate the spacy
import spacy
nlp = spacy.load('en_core_web_sm')


In [ ]:
def predict(model, sentence, word_dict, max_len=512):
    model.eval()
    
    tokens = [tok.text for tok in nlp(sentence)]
    unk_idx = word_dict["<unk>"]
    indexed = [word_dict.get(t, unk_idx) for t in tokens][:max_len]

    seq = torch.tensor(indexed, dtype=torch.long).unsqueeze(0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    seq = seq.to(device)

    with torch.no_grad():
        output = model(seq)     # <-- only ONE argument
        pred = torch.argmax(output, dim=1).item()

    return pred


In [ ]:
#Lets use it for predicting the model

x = predict(model, "how the #altright uses  &amp; insecurity to lure men into #whitesupremacy",word_dict)
print(x)

In [ ]:
#load the csv file
df_test = pd.read_csv(r'twitter-hate-speech/test_tweets_anuFYb8.csv')

#Apply all the steps we did before training
df_test = df_test.drop(['id'], axis=1)
df_test['tweet'] = df_test['tweet'].apply((lambda x: review_to_words(x)))
df_test['tweet'] = df_test['tweet'].apply(lambda x : ' '.join(replace_words[word] if word in replace_words else word for word in x.split()))
df_test['tweet'] = df_test['tweet'].apply(lambda x: x.replace('user',''))

#print the dataframe
df_test.head()

In [ ]:
df_test.describe()

In [ ]:
#we will now make a list of predictions
label_test = [predict(model, df_test['tweet'][ind],word_dict) for ind in df_test.index]

print(label_test)

In [ ]:
#convert the list into the dataframe and assign it to column called label into the test dataframe
df_test['label'] = pd.DataFrame(label_test)

#display the results
df_test.head()

In [ ]:
#write the file into the csv
df_test.to_csv(r'twitter-hate-speech/test_results.csv')

In [ ]:
#a simple visualization to show the prediction
df_test.label.value_counts().plot(kind='barh')
print(df_test.label.value_counts())